# cytozip Python API

### view header

In [1]:
import os,sys
import cytozip as czip
reader=czip.Reader("../czip_example_data/FC_E17a_3C_1-1-I3-F13.cz")
reader.print_header()

magic  :  b'CZIP'
version  :  0.0
total_size  :  30629432
message  :  mm10_with_chrL.allc.cz
formats  :  ['B', 'B']
columns  :  ['mc', 'cov']
dimensions  :  ['chrom']
header_size  :  59


In [2]:
reader.header

{'magic': b'CZIP',
 'version': 0.0,
 'total_size': 30629432,
 'message': 'mm10_with_chrL.allc.cz',
 'formats': ['B', 'B'],
 'columns': ['mc', 'cov'],
 'dimensions': ['chrom'],
 'header_size': 59}

### summary chunks & blocks

In [3]:
df=reader.summary_chunks(printout=False)
df

,chrom,chunk_start_offset,chunk_size,chunk_tail_offset,chunk_nblocks,chunk_nrows
chunk_dims,,,,,,
"(chr1,)",chr1,59,2274268,2293628,2410,78962721
"(chr10,)",chr10,2293628,1479939,3786437,1606,52609184
"(chr11,)",chr11,3786437,1550759,5349922,1588,52027265
"(chr12,)",chr12,5349922,1376780,6738644,1490,48799752
"(chr13,)",chr13,6738644,1369196,8119766,1488,48750883
...,...,...,...,...,...,...
"(chrUn_GL456379,)",chrUn_GL456379,30590507,563,30591109,1,26755
"(chrUn_GL456366,)",chrUn_GL456366,30591109,408,30591556,1,16990
"(chrUn_GL456368,)",chrUn_GL456368,30591556,253,30591848,1,7478


Every chunk has a dimension (chrom, sample, cell types or the combination of those dimensions)

In [4]:
df=reader.summary_blocks(printout=False)
df

,chunk_dims,block_start_offset,block_size,block_data_start,block_data_len
0,"(chr1,)",69,1056,0,65535
1,"(chr1,)",1125,1034,65535,65535
2,"(chr1,)",2159,896,131070,65535
3,"(chr1,)",3055,932,196605,65535
4,"(chr1,)",3987,1011,262140,65535
...,...,...,...,...,...
33764,"(chrUn_GL456366,)",30591119,398,0,33980
33765,"(chrUn_GL456368,)",30591566,243,0,14956
33766,"(chrUn_JH584304,)",30591858,11170,0,65535
33767,"(chrUn_JH584304,)",30603028,13290,65535,30989


### query

In [5]:
?reader.query

Signature:
reader.query(
    dimension=None,
    start=None,
    end=None,
    regions=None,
    query_col=[0],
    reference=None,
    printout=True,
)
Docstring:
query .cz file by dimension, start and end, if regions provided, dimension, start and
end should be None, regions should be a list, each element of regions
is a list, for example, regions=[[('cell1','chr1'),1,10],
[('cell10','chr22'),100,200]],and so on.

Parameters
----------
dimension : str
        dimension, such as chr1 (if dimensions 'chrom' is inlcuded in
        header['dimensions']), or sample1 (if something like 'sampleID' is
        included in header['dimensions'] and chunk contains such dimension)
start : int
        start position, if None, the entire dimension would be returned.
end : int
        end position
regions : list or file path
        regions=[
        [dimension,start,end], [dimension,start,end]
        ],
        dimension is a tuple, for example, dimension=('chr1');
        if regions is a file pat

In [6]:
for record in reader.query(dimension="chr9",start=3000294,end=3000472,
                           reference=os.path.expanduser("~/Ref/mm10/annotations/mm10_with_chrL.allc.cz"),
                           printout=False):
    print(record) # list

['chr9', '3000294', '-', 'CAT', '54', '63']
['chr9', '3000296', '+', 'CCT', '0', '30']
['chr9', '3000297', '+', 'CTA', '1', '31']
['chr9', '3000300', '+', 'CAA', '1', '29']
['chr9', '3000304', '-', 'CAT', '0', '71']
['chr9', '3000305', '-', 'CCA', '1', '72']
['chr9', '3000307', '+', 'CAT', '1', '29']
['chr9', '3000312', '+', 'CTA', '1', '30']
['chr9', '3000321', '+', 'CCA', '0', '33']
['chr9', '3000322', '+', 'CAA', '0', '29']
['chr9', '3000325', '+', 'CTT', '2', '31']
['chr9', '3000331', '+', 'CAG', '0', '35']
['chr9', '3000333', '-', 'CTG', '1', '91']
['chr9', '3000338', '+', 'CCT', '0', '34']
['chr9', '3000339', '+', 'CTC', '0', '34']
['chr9', '3000341', '+', 'CGC', '32', '36']
['chr9', '3000342', '-', 'CGA', '69', '85']
['chr9', '3000343', '+', 'CCA', '1', '37']
['chr9', '3000344', '+', 'CAT', '1', '38']
['chr9', '3000351', '+', 'CAC', '0', '44']
['chr9', '3000353', '+', 'CGT', '36', '45']
['chr9', '3000354', '-', 'CGT', '77', '82']
['chr9', '3000356', '+', 'CCT', '1', '46']
['chr9

### Pack .allc.tsv.gz to .cz without coordinates (using reference)

In [7]:
? czip.allc.allc2cz

Signature:
 czip.allc.allc2cz(
    input,
    outfile,
    reference=None,
    missing_value=[0, 0],
    formats=['B', 'B'],
    columns=['mc', 'cov'],
    dimensions=['chrom'],
    usecols=[4, 5],
    pr=0,
    pa=1,
    sep='\t',
    path_to_chrom=None,
    chunksize=5000,
)
Docstring:
convert allc.tsv.gz to .cz file.

Parameters
----------
input : path
    path to allc.tsv.gz, should has .tbi index.
outfile : path
    output .cz file
reference : path
    path to reference coordinates.
formats: list
    When reference is provided, we only need to pack mc and cov,
    ['H', 'H'] is suggested (H is unsigned short integer, only 2 bytes),
    if reference is not provided, we also need to pack position (Q is
    recommanded), in this case, formats should be ['Q','H','H'].
columns: list
    columns names, in default is ['mc','cov'] (reference is provided), if no
    referene provided, one should use ['pos','mc','cov'].
dimensions: list
    dimensions passed to cytozip.Writer, dimension nam

```python
czip.allc.allc2cz("FC_E17a_3C_1-1-I3-F13.allc.tsv.gz", outfile="FC_E17a_3C_1-1-I3-F13.cz",
                  reference="~/Ref/mm10/annotations/mm10_with_chrL.allc.cz")
```

## Convert .cz into allc using Python API

In [8]:
reader=czip.Reader("../czip_example_data/FC_E17a_3C_1-1-I3-F13.cz")

In [9]:
time reader.to_allc(output="test.allc.tsv.gz", reference="~/Ref/mm10/annotations/mm10_with_chrL.allc.cz", dimension=None)

CPU times: user 1min 43s, sys: 9.09 s, total: 1min 52s
Wall time: 1min 52s


## Remote Reading (from URL)
cytozip files with a chunk index can be read directly from HTTP/HTTPS URLs, without downloading the entire file. This uses HTTP Range requests to fetch only the required chunks. The speed of querying remote cz file is almost the same as the local cz file.

Key features:
- **RemoteFile**: File-like object backed by HTTP Range requests with 2MB read-ahead cache
- **Reader.from_url()**: Open a remote .cz file with O(1) chunk lookup via chunk index
- **Auto-detection**: Passing a URL to `Reader()` automatically uses remote mode

To enable CORS on Apache2 under CentOS, add the following to /etc/httpd/conf/httpd.conf
```shell
Header set Access-Control-Allow-Origin "*"
Header set Access-Control-Allow-Methods "GET, POST, OPTIONS, PUT"
Header set Access-Control-Allow-Headers "Content-Type, Authorization, Range"
Header set Access-Control-Expose-Headers "Content-Range, Accept-Ranges, Content-Length"
Header set Accept-Ranges "bytes"
LoadModule headers_module modules/mod_headers.so
```
Test whether a file support range request:
```shell
curl -I -H "Range: bytes=0-999" http://your-server.com/yourfile.bam
```

### Read remote cz file from webserver

In [ ]:
import cytozip as czip
# url = "http://gale-sapiens.salk.edu/bican/FC_E17a_3C_1-1-I3-F13.cz"
url = "http://gale-sapiens.salk.edu/bican/UCI2424_CX1718BS0102_MGM1_1_P10-1-I17-A16.with_coordinate.cz"

# Method 1: Using Reader.from_url (recommended)
reader = czip.Reader.from_url(url)

# Method 2: Pass URL directly to Reader (auto-detected)
# reader = czip.Reader(url)

# Once opened, all Reader methods work the same as local files:
reader.print_header()
reader.summary_chunks()
for record in reader.fetch(("chr1",)):
    print(record)
    break
for record in reader.query(dimension="chr11", start=51411014,end=51411042, printout=False):
    print(record)
reader.close()

print("Remote reading requires an HTTP server supporting Range requests.")
print("The .cz file should include a chunk index for optimal performance.")

### Read remote cz file from figshare

In [ ]:
import cytozip as czip
# if you host your remote .cz file on figshare:
url="https://figshare.com/ndownloader/files/63531984" 
# UCI2424_CX1718BS0102_MGM1_1_P10-1-I17-A16.with_coordinate.cz
import requests
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) "
                    "Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://figshare.com/",
    "Accept": "*/*",
})
session.get("https://figshare.com")  # acquire cookies
reader = czip.Reader.from_url(url,session=session)
print(reader.header)
print(reader.summary_chunks())
for record in reader.query(dimension="chr9",start=60610139,end=60610151,
                           printout=False):
    print(record) # list

In [ ]:
import os
url="https://figshare.com/ndownloader/files/63531981" 
# FC_E17a_3C_1-1-I3-F13.cz: no coordinates stored in this cz file
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) "
                    "Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://figshare.com/",
    "Accept": "*/*",
})
session.get("https://figshare.com")  # acquire cookies
reader = czip.Reader.from_url(url,session=session)
print(reader._is_remote)
print(reader.header)
print(reader.summary_chunks())

# query with reference
for record in reader.query(dimension="chr9",start=3000294,end=3000472,
                           reference=os.path.expanduser("~/Ref/mm10/annotations/mm10_with_chrL.allc.cz"),
                           printout=False):
    print(record) # list